# Notebook 03 — Momentum Signal Construction and IC Analysis

**Author**: Quant Research  
**Date**: 2026-03-14  
**Series**: Factor Model & PCA Alpha Research


## Executive Summary

This notebook implements the **12-1 month cross-sectional momentum** signal of
Jegadeesh & Titman (1993) and evaluates its predictive power using the
**Information Coefficient (IC)** framework from Grinold & Kahn (1999).

We also implement **short-term reversal** (1-month return) and compare the two
signals. The juxtaposition is informative: momentum operates at 1–12 month
horizons, while reversal operates at 1–4 week horizons — they represent
fundamentally different behavioral phenomena.

Key analyses:
1. **Signal construction** — 12-1 momentum and 1-month reversal
2. **IC computation** — rank correlation of signal with forward 1-month return
3. **Rolling IC plot** — signal consistency over time
4. **IC summary table** — mean IC, IC t-stat, IC IR (the practitioner's key metrics)
5. **IC decay analysis** — signal half-life at 5/10/21/63 day horizons

**References**:  
Jegadeesh, N. & Titman, S. (1993). *Journal of Finance*, 48(1), 65–91.  
Grinold, R. & Kahn, R. (1999). *Active Portfolio Management*, 2nd ed. McGraw-Hill.


## 1. Imports and Setup


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from scipy.stats import spearmanr, ttest_1samp
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

SRC_AVAILABLE = False
try:
    from src.data.loader import fetch_returns
    from src.data.universe import get_universe
    from src.factors.technical import momentum_12_1, short_term_reversal
    from src.factors.combine import compute_ic, ic_summary, ic_decay
    from src.utils import winsorize_and_zscore
    SRC_AVAILABLE = True
    print("[OK] src modules loaded")
except ImportError as e:
    print(f"[WARN] src import failed: {e} — using inline implementations")

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

START = "2010-01-01"
END   = "2023-12-31"
print(f"Period: {START} → {END}")


## 2. Load Return Data and Define Helper Functions

If the data pipeline is unavailable, we simulate a return panel with mild
momentum and reversal effects embedded — allowing us to verify the signal
construction produces meaningful ICs.


In [ ]:
# ─── Inline implementations (used if src/ unavailable) ────────────────────────

def winsorize_zscore(series: pd.Series, n_std: float = 3.0) -> pd.Series:
    """Winsorize at ±n_std then z-score. Applied cross-sectionally."""
    mu, sigma = series.mean(), series.std()
    clipped = series.clip(lower=mu - n_std * sigma, upper=mu + n_std * sigma)
    return (clipped - clipped.mean()) / clipped.std().replace(0, np.nan)

def compute_momentum_12_1(prices: pd.DataFrame) -> pd.DataFrame:
    """
    12-1 month cross-sectional momentum (Jegadeesh & Titman 1993).
    momentum[t] = cumulative return from t-252 to t-21 (skip last month).
    Applied cross-sectionally and winsorized at ±3σ.
    """
    log_ret = np.log(1 + prices)  # treat as daily returns
    mom = log_ret.rolling(252 - 21).sum().shift(21)  # skip last month
    return mom.apply(winsorize_zscore, axis=1)

def compute_reversal_1m(prices: pd.DataFrame) -> pd.DataFrame:
    """
    Short-term reversal: negative of the past 21-day return.
    The sign flip makes high values indicate expected outperformers
    (i.e., recent losers expected to reverse).
    """
    log_ret = np.log(1 + prices)
    rev = -log_ret.rolling(21).sum()
    return rev.apply(winsorize_zscore, axis=1)

def compute_ic_series(signal: pd.DataFrame, fwd_returns: pd.DataFrame,
                      horizon: int = 21) -> pd.Series:
    """
    Rank IC (Spearman correlation) between signal and forward returns.
    IC[t] = spearmanr(signal[t], returns[t:t+horizon]).
    """
    fwd = fwd_returns.shift(-horizon).rolling(horizon).sum()
    ic_vals = []
    dates = []
    for date in signal.index:
        sig_row = signal.loc[date].dropna()
        if date not in fwd.index:
            continue
        fwd_row = fwd.loc[date].dropna()
        common = sig_row.index.intersection(fwd_row.index)
        if len(common) < 20:
            continue
        rho, _ = spearmanr(sig_row[common], fwd_row[common])
        ic_vals.append(rho)
        dates.append(date)
    return pd.Series(ic_vals, index=dates, name='IC')

def summarize_ic(ic: pd.Series) -> dict:
    """Grinold & Kahn IC summary statistics."""
    return {
        'Mean IC':     ic.mean(),
        'Std IC':      ic.std(),
        'IC IR':       ic.mean() / ic.std() if ic.std() > 0 else np.nan,
        't-stat':      ttest_1samp(ic.dropna(), 0).statistic,
        'p-value':     ttest_1samp(ic.dropna(), 0).pvalue,
        '% Positive':  (ic > 0).mean() * 100,
        'N months':    len(ic.dropna()),
    }


# ─── Simulate or load return data ─────────────────────────────────────────────

def simulate_returns_with_momentum(n_stocks=80, seed=42):
    """Simulate returns with a genuine (but small) momentum effect."""
    rng = np.random.default_rng(seed)
    idx = pd.bdate_range(START, END)
    T, N = len(idx), n_stocks
    base = rng.normal(3e-4, 0.012, (T, N))

    # Inject momentum: yesterday's 6-month winner has slightly higher return today
    returns_mat = base.copy()
    for t in range(126, T):
        past_6m = base[t-126:t].sum(axis=0)
        returns_mat[t] += 0.002 * (past_6m - past_6m.mean()) / past_6m.std()

    tickers = [f'S{i:03d}' for i in range(N)]
    return pd.DataFrame(returns_mat, index=idx, columns=tickers)


SYNTHETIC = False
if SRC_AVAILABLE:
    try:
        universe = get_universe()
        returns = fetch_returns(universe, start=START, end=END)
        print(f"[OK] Live returns: {returns.shape}")
    except Exception as e:
        print(f"[WARN] {e} — using synthetic")
        SYNTHETIC = True
else:
    SYNTHETIC = True

if SYNTHETIC:
    print("[INFO] Generating synthetic returns with embedded momentum signal.")
    returns = simulate_returns_with_momentum()
    print("[SYNTHETIC DATA]")

returns = returns.dropna(axis=1, thresh=int(0.9 * len(returns)))
print(f"Clean return matrix: {returns.shape}")


## 3. Construct Momentum and Reversal Signals

**12-1 month momentum** (Jegadeesh & Titman 1993, equation 1):
$$\text{MOM}_{i,t} = \sum_{d=t-252}^{t-21} r_{i,d}$$

We skip the most recent month (last 21 trading days) because short-term reversal
contaminates the momentum signal at the 1-month horizon. This is the standard
"12-1" construction in the literature.

After computing the raw signal, we **winsorize at ±3σ** and **z-score** cross-sectionally.
Winsorization is non-negotiable: raw momentum scores have fat tails that corrupt
cross-sectional regressions and IC estimates.


In [ ]:
if SRC_AVAILABLE:
    try:
        mom_signal  = momentum_12_1(returns)
        rev_signal  = short_term_reversal(returns)
        print("[OK] Signals from src/")
    except Exception as e:
        print(f"[WARN] {e} — computing inline")
        mom_signal  = compute_momentum_12_1(returns)
        rev_signal  = compute_reversal_1m(returns)
else:
    mom_signal  = compute_momentum_12_1(returns)
    rev_signal  = compute_reversal_1m(returns)

# Reindex to monthly (end-of-month) for IC computation
monthly_idx = pd.bdate_range(START, END, freq='BM')  # business month-end
monthly_idx = monthly_idx[monthly_idx.isin(mom_signal.index)]

mom_monthly = mom_signal.loc[monthly_idx]
rev_monthly = rev_signal.loc[monthly_idx]
ret_monthly = returns.loc[monthly_idx]

print(f"\nMonthly signal observations: {len(mom_monthly)}")
print(f"Momentum signal sample (first row stats):")
print(mom_monthly.iloc[0].describe().round(3))


## 4. Rolling IC Analysis

The IC at month t is the rank (Spearman) correlation between the signal measured
at the end of month t and the forward 1-month return. A positive IC means the
signal correctly ranks stocks: high-signal stocks outperform low-signal stocks.

We compute a **12-month rolling average** of IC to smooth out noise and identify
regime changes in signal efficacy.


In [ ]:
fwd_1m_returns = returns.shift(-21).rolling(21).sum()  # forward 21-day return

if SRC_AVAILABLE:
    try:
        ic_mom = compute_ic(mom_signal, fwd_1m_returns, horizon=21)
        ic_rev = compute_ic(rev_signal, fwd_1m_returns, horizon=21)
        print("[OK] IC from src/")
    except Exception as e:
        print(f"[WARN] {e} — computing inline")
        ic_mom = compute_ic_series(mom_monthly, returns, horizon=21)
        ic_rev = compute_ic_series(rev_monthly, returns, horizon=21)
else:
    ic_mom = compute_ic_series(mom_monthly, returns, horizon=21)
    ic_rev = compute_ic_series(rev_monthly, returns, horizon=21)

print(f"\nMomentum IC series: {len(ic_mom)} observations")
print(f"Reversal  IC series: {len(ic_rev)} observations")

# ── Rolling IC plot ────────────────────────────────────────────────────────────
roll = 12  # 12-month rolling average
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=False)

for ax, (ic, label, color) in zip(axes, [
    (ic_mom, '12-1 Momentum', '#1f77b4'),
    (ic_rev, '1-Month Reversal', '#d62728'),
]):
    ax.bar(ic.index, ic.values, color=color, alpha=0.3, width=20, label='Monthly IC')
    ax.plot(ic.index, ic.rolling(roll).mean(), color=color, linewidth=2.0,
            label=f'{roll}-month rolling mean')
    ax.axhline(0, color='black', linewidth=1.0)
    ax.axhline(ic.mean(), color=color, linewidth=1.2, linestyle='--', alpha=0.7,
               label=f'Full-period mean = {ic.mean():.4f}')
    ax.set_ylabel('Rank IC (Spearman)')
    ax.set_title(f'{label} — Monthly IC with {roll}-Month Rolling Average', fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_ylim(-0.25, 0.25)

plt.tight_layout()
plt.savefig('../results/ic_rolling.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. IC Summary Table

The key metrics from Grinold & Kahn (1999):

- **Mean IC**: Average predictive accuracy. A mean IC of 0.05 is considered good in equities.
- **IC IR** (= Mean IC / Std IC): Signal consistency. IC IR > 0.5 indicates the signal
  works reliably, not just in isolated periods. This is more important than mean IC alone.
- **t-stat**: Statistical significance of mean IC under H₀: IC = 0.


In [ ]:
if SRC_AVAILABLE:
    try:
        summary_mom = ic_summary(ic_mom)
        summary_rev = ic_summary(ic_rev)
    except Exception:
        summary_mom = summarize_ic(ic_mom)
        summary_rev = summarize_ic(ic_rev)
else:
    summary_mom = summarize_ic(ic_mom)
    summary_rev = summarize_ic(ic_rev)

summary_df = pd.DataFrame({
    '12-1 Momentum': summary_mom,
    '1-Month Reversal': summary_rev,
}).T

print("\n=== IC Summary Statistics ===")
print(summary_df.round(4).to_string())
print()
print("Grinold & Kahn benchmarks:")
print("  Mean IC > 0.05  →  useful signal")
print("  IC IR  > 0.50   →  consistent signal")
print("  t-stat > 2.0    →  statistically significant at 5%")


## 6. IC Decay Analysis

The IC decay curve shows how the predictive power of the signal degrades at
longer forward horizons. A signal that shows IC at 1 month but zero IC at 3 months
has a short half-life — requiring high-frequency rebalancing and thus higher
transaction costs.

Momentum is known to peak at 3–12 month horizons; reversal decays rapidly within
the first 4–8 weeks.


In [ ]:
horizons = [5, 10, 21, 63]  # 1-week, 2-week, 1-month, 3-month
horizon_labels = ['5d (1wk)', '10d (2wk)', '21d (1mo)', '63d (3mo)']

decay_results = {'Momentum': {}, 'Reversal': {}}

for h, hlabel in zip(horizons, horizon_labels):
    for signal_name, signal_df in [('Momentum', mom_monthly), ('Reversal', rev_monthly)]:
        fwd_h = returns.shift(-h).rolling(h).sum()
        if SRC_AVAILABLE:
            try:
                ic_h = compute_ic(signal_df, fwd_h, horizon=h)
            except Exception:
                ic_h = compute_ic_series(signal_df, returns, horizon=h)
        else:
            ic_h = compute_ic_series(signal_df, returns, horizon=h)
        decay_results[signal_name][hlabel] = ic_h.mean() if len(ic_h) > 0 else np.nan

decay_df = pd.DataFrame(decay_results)
print("\n=== IC Decay Table (Mean IC by Horizon) ===")
print(decay_df.round(4).to_string())

# ── IC Decay Plot ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(horizons))
width = 0.35

bars1 = ax.bar(x - width/2, decay_df['Momentum'], width, label='12-1 Momentum',
               color='#1f77b4', edgecolor='navy', linewidth=0.5)
bars2 = ax.bar(x + width/2, decay_df['Reversal'],  width, label='1-Month Reversal',
               color='#d62728', edgecolor='darkred', linewidth=0.5)

for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    ypos = h + 0.001 if h >= 0 else h - 0.003
    ax.text(bar.get_x() + bar.get_width()/2, ypos, f'{h:.4f}',
            ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(horizon_labels)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Mean IC')
ax.set_title('IC Decay by Forward Horizon — Momentum vs Reversal', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../results/ic_decay.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Quintile Return Spread — Sanity Check

Beyond IC, we verify the signal using a **quintile sort**: sort stocks into 5 groups
by signal rank, compute average forward 1-month returns per quintile, and confirm
a monotonic spread from Q1 (short) to Q5 (long).

A monotonic spread is necessary (but not sufficient) for a tradeable strategy —
it confirms the signal is not purely driven by outliers.


In [ ]:
fwd_21 = returns.shift(-21).rolling(21).sum()

quintile_means = []
for date in mom_monthly.index[:-1]:  # exclude last month (no forward return)
    if date not in fwd_21.index:
        continue
    sig_row = mom_monthly.loc[date].dropna()
    fwd_row = fwd_21.loc[date].dropna()
    common = sig_row.index.intersection(fwd_row.index)
    if len(common) < 20:
        continue
    combined = pd.DataFrame({'signal': sig_row[common], 'fwd': fwd_row[common]})
    combined['quintile'] = pd.qcut(combined['signal'], 5, labels=[1,2,3,4,5])
    q_means = combined.groupby('quintile')['fwd'].mean()
    quintile_means.append(q_means)

if quintile_means:
    quintile_avg = pd.DataFrame(quintile_means).mean()
    quintile_avg.index = [f'Q{i}' for i in range(1, 6)]

    fig, ax = plt.subplots(figsize=(8, 5))
    colors = ['#d62728', '#ff7f0e', '#bcbd22', '#17becf', '#2ca02c']
    bars = ax.bar(quintile_avg.index, quintile_avg.values * 100,
                  color=colors, edgecolor='black', linewidth=0.5)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylabel('Average 1-Month Forward Return (%)')
    ax.set_title('Quintile Sort — 12-1 Momentum Signal\nQ1=Short, Q5=Long',
                 fontweight='bold')

    spread = (quintile_avg.iloc[-1] - quintile_avg.iloc[0]) * 100
    ax.text(0.98, 0.95, f'Q5-Q1 spread = {spread:.2f}%',
            transform=ax.transAxes, ha='right', va='top',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    plt.tight_layout()
    plt.savefig('../results/momentum_quintile_sort.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\nQ5-Q1 spread: {spread:.4f}%/month")
    print(quintile_avg.apply(lambda x: f'{x*100:.4f}%'))
else:
    print("[WARN] Insufficient data for quintile sort.")


## 8. Key Findings

| Metric | 12-1 Momentum | 1-Month Reversal | Benchmark (G&K) |
|--------|--------------|------------------|------------------|
| Mean IC | See table | See table | ≥ 0.05 |
| IC IR | See table | See table | ≥ 0.50 |
| IC t-stat | See table | See table | ≥ 2.0 |

**Expected findings** based on the literature (Jegadeesh & Titman 1993):
- Momentum IC should be positive with IC IR around 0.4–0.7 in liquid large-caps
- Reversal IC is typically negative at the 1-month forward horizon (recent losers bounce back)
- Both signals degrade at the 3-month horizon but in opposite directions

**Caveats**:
- These results depend on whether live or synthetic data was used
- No transaction cost adjustment here — see Notebook 05 for net-of-cost analysis
- Momentum is subject to crash risk (March 2009, March–June 2020) — this IC analysis
  does not capture tail risk

**Next**: Notebook 04 combines momentum + reversal into a composite alpha signal
using IC-weighted averaging and orthogonalizes against FF factors.


In [ ]:
print("=" * 60)
print("Notebook 03 complete.")
print(f"Synthetic data: {SYNTHETIC}")
print(f"IC observations (momentum): {len(ic_mom)}")
print(f"IC observations (reversal):  {len(ic_rev)}")
print("Outputs saved to ../results/")
print("=" * 60)
